# Quick estimates of the problem-set models
Uses the optimized `blp_optimize/blp_new.jl` (threaded multistart, warm-started contraction). Loose settings for fast estimates: tolerance `1e-4`, few optimizer iterations. Estimates are saved in `models_quick/` (the other model folders are untouched). Start the kernel with several threads (`-t auto`) to run starts in parallel.

In [1]:
using MAT, DataFrames, Statistics, LinearAlgebra, JLD2
include("blp_optimize/blp_new.jl")
using .blp
Threads.nthreads()

1

## Settings

In [2]:
const TOLER = 1e-4      # contraction tolerance (precise runs: 1e-12)
const MAX_ITER = 1000   # contraction iterations
const NS = 1            # starting points
const ITERATIONS = 15   # optimizer iterations per start
const OUTDIR = "models_quick"
mkpath(OUTDIR)

beta = [5, 1, 1]; alpha_bar = -1; gamma = [2, 1, 1]; sigma = 1.0

1.0

## Data

In [3]:
simdir = "../build/input/public/simulation"
d1 = matread(joinpath(simdir, "10markets3products.mat"))
d2 = matread(joinpath(simdir, "100markets3products.mat"))

function process_data(d)
    J, T = size(d["P_opt"])
    p = reshape(d["P_opt"], (T*J, 1)); shares = reshape(d["shares"], (T*J, 1))
    data = blp.BLPData(T, J, d["x1"], p, shares, log.(d["alphas"] .- 1))
    model = blp.BLPDemand(BLPData = data)
    model.xi_real = d["xi_all"]
    model.cost = gamma[1] .+ (gamma[2] * d["w"] + gamma[3] * d["Z"] + d["eta"])
    model.markup .= ((model.p - model.cost) ./ model.p)
    model.profit = (model.p - model.cost) .* model.shares
    return model
end

# Estimates a model with the quick settings and saves it
function estimate!(model, name)
    t = @elapsed if model isa blp.BLPDemand
        blp.estimate_blp!(model; toler = TOLER, max_iter = MAX_ITER, Ns = NS, iterations = ITERATIONS)
    else
        blp.estimate_blp_joint!(model; toler = TOLER, max_iter = MAX_ITER, Ns = NS, iterations = ITERATIONS)
    end
    jldsave(joinpath(OUTDIR, "$name.jdl2"); model = model)
    println("$name: $(round(t, digits = 1))s, params = ", round.(vec(model.params), digits = 4))
    return model
end

timings = Dict{String, Float64}()
set_Z!(m, Z) = (m.Z = Z; m.W = inv(Z' * Z); m)

set_Z! (generic function with 1 method)

In [4]:
## No Hausman instruments

In [5]:
model = process_data(d2)
model.beta .= vcat(beta, [alpha_bar]); model.sigma = sigma
blp.simulate_BLP!(model, delta = model.X_all * model.beta + model.xi_real, sigma = sigma, all = true)

price_bar = blp.compute_hausman(model)
X_rivals = blp.compute_blp(model)
set_Z!(model, hcat(model.X, X_rivals))
timings["model_nh"] = @elapsed model = estimate!(model, "model_nh")

  start sigma0 = 0.922 -> sigma = 1.0543, obj = 0.005147891755673372, 44.3s


┌ Warning: Attempting to store Optim.var"#Fminbox##0#Fminbox##1".
│ JLD2 only stores functions by name.
│  This may not be useful for anonymous functions.
└ @ JLD2 ~/.julia/packages/JLD2/rm5a0/src/data/writing_datatypes.jl:447
┌ Warning: Attempting to store Optim.var"#Fminbox##0#Fminbox##1".
│ JLD2 only stores functions by name.
│  This may not be useful for anonymous functions.
└ @ JLD2 ~/.julia/packages/JLD2/rm5a0/src/data/writing_datatypes.jl:447
┌ Warning: Attempting to store Optim.var"#optimize##0#optimize##1"{Vector{Float64}, Vector{Int64}, Optim.Fminbox{Optim.LBFGS{Nothing, LineSearches.InitialStatic{Float64}, LineSearches.HagerZhang{Float64, Base.RefValue{Bool}}, Returns{Nothing}}, Float64, Optim.var"#Fminbox##0#Fminbox##1"}}.
│ JLD2 only stores functions by name.
│  This may not be useful for anonymous functions.
└ @ JLD2 ~/.julia/packages/JLD2/rm5a0/src/data/writing_datatypes.jl:447


model_nh: 45.5s, params = [5.5905, 1.1001, 1.0817, -1.1689, 1.0543]


50.058247959

In [6]:
model_ls = process_data(d1)
p_bar_ls = blp.compute_hausman(model_ls); X_rivals_ls = blp.compute_blp(model_ls)
set_Z!(model_ls, hcat(model_ls.X, X_rivals_ls))
timings["model_ls_nh"] = @elapsed model_ls = estimate!(model_ls, "model_ls_nh")

Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached ma

┌ Warning: Attempting to store Optim.var"#Fminbox##0#Fminbox##1".
│ JLD2 only stores functions by name.
│  This may not be useful for anonymous functions.
└ @ JLD2 ~/.julia/packages/JLD2/rm5a0/src/data/writing_datatypes.jl:447
┌ Warning: Attempting to store Optim.var"#Fminbox##0#Fminbox##1".
│ JLD2 only stores functions by name.
│  This may not be useful for anonymous functions.
└ @ JLD2 ~/.julia/packages/JLD2/rm5a0/src/data/writing_datatypes.jl:447
┌ Warning: Attempting to store Optim.var"#optimize##0#optimize##1"{Vector{Float64}, Vector{Int64}, Optim.Fminbox{Optim.LBFGS{Nothing, LineSearches.InitialStatic{Float64}, LineSearches.HagerZhang{Float64, Base.RefValue{Bool}}, Returns{Nothing}}, Float64, Optim.var"#Fminbox##0#Fminbox##1"}}.
│ JLD2 only stores functions by name.
│  This may not be useful for anonymous functions.
└ @ JLD2 ~/.julia/packages/JLD2/rm5a0/src/data/writing_datatypes.jl:447


2.127260542

In [7]:
model_inc = deepcopy(model)
set_Z!(model_inc, hcat(model_inc.X_all, X_rivals))
timings["model_inc_nh"] = @elapsed model_inc = estimate!(model_inc, "model_inc_nh")

Reached max number of iterations
  start sigma0 = 0.922 -> sigma = 0.9893, obj = 0.0057456633363205944, 6.5s


┌ Warning: Attempting to store Optim.var"#Fminbox##0#Fminbox##1".
│ JLD2 only stores functions by name.
│  This may not be useful for anonymous functions.
└ @ JLD2 ~/.julia/packages/JLD2/rm5a0/src/data/writing_datatypes.jl:447
┌ Warning: Attempting to store Optim.var"#Fminbox##0#Fminbox##1".
│ JLD2 only stores functions by name.
│  This may not be useful for anonymous functions.
└ @ JLD2 ~/.julia/packages/JLD2/rm5a0/src/data/writing_datatypes.jl:447


model_inc_nh: 6.9s, params = [4.7745, 0.9217, 1.0215, -0.9037, 0.9893]


┌ Warning: Attempting to store Optim.var"#optimize##0#optimize##1"{Vector{Float64}, Vector{Int64}, Optim.Fminbox{Optim.LBFGS{Nothing, LineSearches.InitialStatic{Float64}, LineSearches.HagerZhang{Float64, Base.RefValue{Bool}}, Returns{Nothing}}, Float64, Optim.var"#Fminbox##0#Fminbox##1"}}.
│ JLD2 only stores functions by name.
│  This may not be useful for anonymous functions.
└ @ JLD2 ~/.julia/packages/JLD2/rm5a0/src/data/writing_datatypes.jl:447


6.858178917

In [ ]:
error 
mistake

## Baseline: T = 100, J = 3 (Hausman price instrument + rival characteristics)

In [ ]:
model = process_data(d2)
model.beta .= vcat(beta, [alpha_bar]); model.sigma = sigma
blp.simulate_BLP!(model, delta = model.X_all * model.beta + model.xi_real, sigma = sigma, all = true)

price_bar = blp.compute_hausman(model)
X_rivals = blp.compute_blp(model)
set_Z!(model, hcat(model.X, price_bar, X_rivals))
timings["model"] = @elapsed model = estimate!(model, "model")

## T = 10, J = 3

In [ ]:
model_ls = process_data(d1)
p_bar_ls = blp.compute_hausman(model_ls); X_rivals_ls = blp.compute_blp(model_ls)
set_Z!(model_ls, hcat(model_ls.X, p_bar_ls, X_rivals_ls))
timings["model_ls"] = @elapsed model_ls = estimate!(model_ls, "model_ls")

## Price as its own instrument

In [ ]:
model_inc = deepcopy(model)
set_Z!(model_inc, hcat(model_inc.X_all, price_bar, X_rivals))
timings["model_inc"] = @elapsed model_inc = estimate!(model_inc, "model_inc")

## Adding the cost shifter w as a demand instrument

In [ ]:
model_ss = deepcopy(model)
set_Z!(model_ss, hcat(model_ss.X, price_bar, X_rivals, d2["w"]))
timings["model_ss"] = @elapsed model_ss = estimate!(model_ss, "model_ss")

model_ls_ss = deepcopy(model_ls)
set_Z!(model_ls_ss, hcat(model_ls_ss.X, p_bar_ls, X_rivals_ls, d1["w"]))
timings["model_ls_ss"] = @elapsed model_ls_ss = estimate!(model_ls_ss, "model_ls_ss")

## Joint demand + supply under three conduct assumptions

In [ ]:
X_supply = hcat(ones(model.T * model.J, 1), d2["Z"], d2["w"])
Z_supply = hcat(model.X, X_rivals, X_supply[:, 2:end])
model_supply = blp.BLPSupply(X = X_supply, Z = Z_supply, W = inv(Z_supply' * Z_supply))

systems = Dict{String, blp.BLP}()
for (name, Omega) in (("bert", Matrix{Float64}(I, model.J, model.J)), ("coll", ones(model.J, model.J)), ("comp", zeros(model.J, model.J)))
    dem = deepcopy(model_ss)
    dem.Omega = Omega
    sys = blp.BLP(BLPDemand = dem, BLPSupply = deepcopy(model_supply))
    timings["system_$name"] = @elapsed estimate!(sys, "model_system_$name")
    systems[name] = sys
end

## Summary

In [ ]:
summary = DataFrame(
    model = ["model", "model_ls", "model_inc", "model_ss", "model_ls_ss"],
    sigma = [m.params[end] for m in (model, model_ls, model_inc, model_ss, model_ls_ss)],
    price_coef = [m.beta[end] for m in (model, model_ls, model_inc, model_ss, model_ls_ss)],
    seconds = [round(timings[n], digits = 1) for n in ("model", "model_ls", "model_inc", "model_ss", "model_ls_ss")])

In [ ]:
summary_sys = DataFrame(
    model = ["bert", "coll", "comp"],
    sigma = [systems[n].params[end] for n in ("bert", "coll", "comp")],
    price_coef = [systems[n].BLPDemand.beta[end] for n in ("bert", "coll", "comp")],
    seconds = [round(timings["system_$n"], digits = 1) for n in ("bert", "coll", "comp")])
summary, summary_sys